In [0]:
STORAGE_ACCOUNT_NAME = "stclickstreamstu"
CONTAINER_NAME = "datalake"

# Nguồn đọc (Silver)
SILVER_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/events"

# Đích đến (Gold - Star Schema 5 Dim, 1 Fact)
GOLD_FACT_EVENTS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/fact_events"
GOLD_DIM_USERS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_users"
GOLD_DIM_PRODUCTS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_products"
GOLD_DIM_DATE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_date"
GOLD_DIM_LOCATION_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_location"
GOLD_DIM_SESSIONS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_sessions"
CHECKPOINT_PATH_GOLD = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/gold_advanced_v2"

print("Đã load xong toàn bộ đường dẫn cho Star Schema!")

In [0]:
# 1. Bảng Dimensions
spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{GOLD_DIM_USERS_PATH}` (user_id STRING, device_type STRING, os STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{GOLD_DIM_PRODUCTS_PATH}` (product_id STRING, category STRING) USING DELTA")

# Đường dẫn mới cho các Dim bổ sung
GOLD_DIM_DATE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_date"
GOLD_DIM_LOCATION_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_location"
GOLD_DIM_SESSIONS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_sessions"

spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{GOLD_DIM_DATE_PATH}` (date_key INT, full_date DATE, year INT, month INT, day INT, hour INT) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{GOLD_DIM_LOCATION_PATH}` (location_id STRING, country STRING, city STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{GOLD_DIM_SESSIONS_PATH}` (session_id STRING, utm_source STRING) USING DELTA")

# 2. Bảng Fact Trung tâm (Đã được tối ưu hóa, chỉ chứa ID và Số liệu)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{GOLD_FACT_EVENTS_PATH}` (
        event_id STRING, 
        date_key INT, 
        location_id STRING, 
        user_id STRING, 
        session_id STRING, 
        product_id STRING, 
        action STRING, 
        price DOUBLE, 
        quantity INT, 
        discount INT, 
        net_revenue DOUBLE
    ) USING DELTA
""")

print("Đã tạo xong 5 bảng Dim và 1 bảng Fact")

In [0]:
silver_stream_df = (spark.readStream
    .format("delta")
    .load(SILVER_TABLE_PATH)
)

In [0]:
def process_gold_advanced(microBatchDF, batchId):
    # Tạo View tạm từ dữ liệu Streaming
    microBatchDF.createOrReplaceTempView("silver_batch")

    # --- 1. XỬ LÝ CÁC BẢNG DIMENSIONS (MERGE) ---
    
    # DIM DATE
    spark.sql(f"""
        MERGE INTO delta.`{GOLD_DIM_DATE_PATH}` AS t
        USING (
            SELECT DISTINCT 
                cast(date_format(event_timestamp, 'yyyyMMdd') as INT) as date_key,
                to_date(event_timestamp) as full_date,
                year(event_timestamp) as year,
                month(event_timestamp) as month,
                day(event_timestamp) as day,
                hour(event_timestamp) as hour
            FROM silver_batch
        ) AS s ON t.date_key = s.date_key
        WHEN NOT MATCHED THEN INSERT *
    """)

    # DIM LOCATION
    spark.sql(f"""
        MERGE INTO delta.`{GOLD_DIM_LOCATION_PATH}` AS t
        USING (
            SELECT DISTINCT md5(concat(country, city)) as location_id, country, city 
            FROM silver_batch WHERE country IS NOT NULL
        ) AS s ON t.location_id = s.location_id
        WHEN NOT MATCHED THEN INSERT *
    """)

    # DIM SESSIONS
    spark.sql(f"""
        MERGE INTO delta.`{GOLD_DIM_SESSIONS_PATH}` AS t
        USING (SELECT DISTINCT session_id, utm_source FROM silver_batch WHERE session_id IS NOT NULL) AS s 
        ON t.session_id = s.session_id
        WHEN NOT MATCHED THEN INSERT *
    """)

    # DIM USERS
    spark.sql(f"""
        MERGE INTO delta.`{GOLD_DIM_USERS_PATH}` AS target
        USING latest_batch AS source
        ON target.user_id = source.user_id
        WHEN MATCHED THEN 
            UPDATE SET target.device_type = source.device_type, 
                       target.os = source.os, 
                       target.last_updated = source.event_timestamp
        WHEN NOT MATCHED THEN 
            INSERT (user_id, device_type, os, last_updated) 
            VALUES (source.user_id, source.device_type, source.os, source.event_timestamp)
    """)

# --- 3. SCD TYPE 2 CHO DIM_PRODUCTS (Lưu lịch sử) ---
    # Thuật toán: 
    # Bước A: Đóng những dòng cũ nếu category bị thay đổi
    spark.sql(f"""
        MERGE INTO delta.`{GOLD_DIM_PRODUCTS_PATH}` AS target
        USING latest_batch AS source
        ON target.product_id = source.product_id AND target.is_current = true
        WHEN MATCHED AND target.category <> source.category THEN
            UPDATE SET target.is_current = false, target.end_date = source.event_timestamp
    """)
    
    # Bước B: Chèn dòng mới cho những sản phẩm chưa có hoặc vừa bị đổi category
    # Chúng ta chỉ chèn nếu sản phẩm đó chưa tồn tại một dòng 'is_current = true' nào
    spark.sql(f"""
        INSERT INTO delta.`{GOLD_DIM_PRODUCTS_PATH}`
        SELECT 
            source.product_id, 
            source.category, 
            source.event_timestamp as start_date, 
            NULL as end_date, 
            true as is_current
        FROM latest_batch source
        WHERE NOT EXISTS (
            SELECT 1 FROM delta.`{GOLD_DIM_PRODUCTS_PATH}` target 
            WHERE target.product_id = source.product_id AND target.is_current = true
        )
    """)

    # --- FACT (INSERT ONLY & CALCULATE METRICS)
    spark.sql(f"""
        INSERT INTO delta.`{GOLD_FACT_EVENTS_PATH}`
        SELECT 
            event_id,
            cast(date_format(event_timestamp, 'yyyyMMdd') as INT) as date_key,
            md5(concat(country, city)) as location_id,
            user_id,
            session_id,
            product_id,
            action,
            price,
            quantity,
            discount,
            (price * quantity) - discount as net_revenue
        FROM silver_batch
    """)

In [0]:
print("Đang Upsert (MERGE) vào 6 bảng lớp Gold...")

query_gold = (silver_stream_df.writeStream
    .foreachBatch(process_gold_advanced)
    .option("checkpointLocation", CHECKPOINT_PATH_GOLD)
    .trigger(availableNow=True)
    .start()
)

query_gold.awaitTermination()
print("Pipeline Star Schema đã chạy hoàn tất!")